# AutoMPG Project
## Exploratory analysis
### Prediction of fuel efficiency given 8 variables

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
import seaborn as sns
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

<div class="alert alert-block alert-info">
<b>Import data</b>
</div>

In [ ]:
# Import data
df = pd.read_csv('../data/raw/auto-mpg.data-original', header = None, names = ['mpg', 'cylinders', 'displacement',
                                                                               'horsepower', 'weight', 'acceleration', 
                                                                               'model_year', 'origin', 'car_name'],
                 index_col=None, usecols = [0, 1, 2, 3, 4, 5, 6, 7, 8], delim_whitespace=True, quotechar='"',
                 skipinitialspace=True)
df.head(20)
print(df.shape)

The dataset read from csv file has 406 rows and 9 columns.
The data dictionary specifies 398 rows and 9 columns.
We therefore have 8 too many rows. Let us see why this is the case.


<div class="alert alert-block alert-info">
<b>Explore data</b>
</div>

In [ ]:
# Type of data
print(df.info())

We have eight numeric variables and one categorical variable.

In [ ]:
# Null values
print(df.isnull().sum())


We have 14 null values. Let's see which records contain null values.

In [ ]:
# Null values
df[df.isnull().any(axis=1)]

The null values are not overlapping. Eight of the records have null values for 'mpg'. These were
probably the ones removed from the dataset used to build the data dictionary - makes sense as we cannot use these
build a predictive model for mpg. Next we remove these values from the dataset. We retain the original dataset,
we output data to the 'interim' data directory.

In [ ]:
df_fin = df.copy()
df_fin.dropna(subset=['mpg'], inplace=True)
print("Number of null values in df_fin: {}".format(df_fin['mpg'].isnull().sum()))
print("Dataframe dimension: {}".format(df_fin.shape))

We now have a dataset with 398 record, as per the data dictionary, there are still 6 missing values in the horsepower
field. We will substitute these with median values.

In [ ]:
median = df_fin['horsepower'].median()
df_fin['horsepower'].fillna(median, inplace=True)
print("Number of null values in horsepower column: {}".format(df_fin['horsepower'].isnull().sum()))
print("Dataframe dimension: {}".format(df_fin.shape))

Now our set is ready for further analysis. We will now look at the distribution of variables and any possible outliers
or heavy tailed distributions.

In [ ]:
print(df_fin.nunique())

There are no columns with only one value. We therefore retain all columns. Interesting to observe that there aren't
many cars of the same type in the sample, although most cars have the same origin. There is also a fair amount of
similarity in mpg which is surprising. The following variables should be considered as categorical:
- origin (nominal)
- model_year (ordinal)
- cylinders (ordinal)

It will make sense to one-hot-encode the origin variable. We will do this later.
Next we look at the distribution of the data.
We start by splitting the response and the features.

In [ ]:
# Split data into features (X) and response (y)
# Predict mpg vs other variables (excluding car name)
X = df_fin.loc[:, ('cylinders', 'displacement','horsepower', 'weight', 'acceleration', 'model_year', 'origin',
                    'car_name')]
y = df_fin.loc[:, 'mpg']

Let us look at the response first.

In [ ]:
y.describe()
_ = y.plot(kind='box', subplots=False);
plt.show();
#y.plot(kind='box', subplots=True, layout=(3,3), sharex=False, sharey=False)
_ = y.plot(kind='density', subplots=False);
plt.show();
_ = y.hist();
plt.show();
skew = y.skew()
print("Skewness value: {}".format(skew))

All looks good here. No obvious issues with the data. The one outlier at 46 is hard to believe for a car
made in the mid - eighties, but who knows, maybe it was an early hybrid prototype. Else the spread is what is expected.
The data is slightly skewed, but still approximately symmetric. No need for any normalisation (for regression purposes).

Let us look at the all the variables together. The distributions of all the datasets look fairly normal.

In [ ]:
df_fin.describe()

As can be seen the distributions of all the datasets look fairly normal. Let us now look at these visually.

In [ ]:
df_fin_num = df_fin.copy()
df_fin_num = df_fin_num.iloc[:, 0:8]
names = ['mpg', 'cylinders', 'displacement','horsepower', 'weight', 'acceleration', 'model_year', 'origin']
df_fin_num.plot(kind='box', subplots=True, layout=(3,3), sharex=False, sharey=False)
plt.show();

In [ ]:
correlations = df_fin_num.corr()
# plot correlation matrix
fig = plt.figure()
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)
ticks = np.arange(0,8,1)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(names)
ax.set_yticklabels(names)
plt.show();

In [ ]:
df_fin_num.plot(kind='density', subplots=True, layout=(3,3), sharex=False)
pd.plotting.scatter_matrix(df_fin_num)
#sns.pairplot(df_fin_num, vars=names, hue="mpg", plot_kws = {'alpha': 0.3, 's': 40, 'edgecolor': 'k'}, palette=sns.color_palette(), size = 9)
plt.show();


No surprises here with regards to the feature variables. It should be noted that there are quite a number of outliers
for the horsepower and acceleration variables. It does make sense however that cars with high horsepower will also
have high acceleration, so there is correlation here in all likelihood.
The correlation plot shows that mpg is strongly inversely correlated to No. of cylinders, displacement, horsepower and
weight. Again this makes sense as cars with more powerful engines will typically have lower mpg (more fuel consumption).
Likewise there is a small positive correlation between mpg and acceleration, model_year and origin. This implies that
faster and newer cars have higher fuel consumption.
The distributions of the feature variables have varying scales, so standardisation would be required for ML purposes.

<div class="alert alert-block alert-info">
<b>Build Model</b>
</div>

We can now start building our model.
We begin with a simple Multi Layer Perceptron using scikit-learn to better understand the data.

In [ ]:
# Put the response y into an array
y = np.ravel(y)

We start by One Hot Encoding the origin variable we found to have insufficient levels
for a continuous (interval) variable.
We leave the car_name variable out for now. We expect there to be a very strong correlation between car model and mpg,
in fact all cars of the same model typically have approximately the same MPG. This fact would however not be useful
for any practical application of our model.
For interest however, we could do a clustering exercise and do a target encoding on the resulting clusters for the
car model variable, seeing as it is categorical with possibly too many categories to one hot encode effectively i.e.
more than 300 classes.

In [ ]:
# One hot encoding
df_fin_nn = df_fin_num.loc[:, ('cylinders', 'displacement','horsepower', 'weight', 'acceleration', 'model_year')]
ohe = OneHotEncoder(categories='auto')
x_origin = ohe.fit_transform(df_fin.origin.values.reshape(-1, 1)).toarray()
x_origin_ohe = pd.DataFrame(x_origin, columns = ["origin_"+str(int(i)) for i in range(x_origin.shape[1])])

In [ ]:
# Add encoded feature to the dataframe
df_fin_nn.reset_index(drop=True, inplace=True)
x_origin_ohe.reset_index(drop=True, inplace=True)
df_fin_nn = pd.concat([df_fin_nn, x_origin_ohe], axis=1)

X = df_fin_nn.copy()
X.head()

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

In [ ]:
# Scale data
scaler = StandardScaler()

# Fit on training data set
scaler.fit(X_train)
X_train = scaler.transform(X_train)

# Apply to test data
X_test = scaler.transform(X_test)

### MSE before regularisation

In [ ]:
# Fit and check MSE before regularisation
reg = MLPRegressor(max_iter=50000, solver="adam", activation="tanh", hidden_layer_sizes=(5, 5), random_state=1)
reg.fit(X_train, y_train)

# Predict
y_pred = reg.predict(X_test)

# MSE before regularisation
mean_squared_error(y_pred,y_test)

Optimise neural network with regularisation

In [ ]:
# Select range over which to find regularisation parameter - exponential used for even distribution of values
reg_par = [np.e**n for n in np.arange(-3,6,1)]

validation_scores = {}
print(" alpha  |  Sq.Error")
for param in reg_par:
    reg = MLPRegressor(max_iter=1000000, solver="adam", activation="tanh", hidden_layer_sizes=(5, 5), alpha=param, random_state=1)
    score = cross_val_score(estimator=reg, X=X_train, y=y_train, cv=2, scoring="neg_mean_squared_error")
    validation_scores[param] = -score.mean()
    print("%0.5f |  %s" % (param, -score.mean()))

# Plot the error function against regularisation parameter
plt.plot([np.log(i) for i in validation_scores.keys()], list(validation_scores.values()));
plt.xlabel("Ln of alpha");
plt.ylabel("Mean Sq Error");

The lowest cross-validation error and hence the value to use for the `alpha` parameter is as follows.

In [ ]:
min_score = ([np.log(name) for name, score in validation_scores.items() if score==min(validation_scores.values())][0])
# Find lowest value.
print("The lowest cross validation error is: %s" % (min(validation_scores.values())))
print("This corresponds to regularisation parameter e**%s" % min_score)

### MSE after regularisation

In [ ]:
# Fit data with the best parameter
reg = MLPRegressor(max_iter=1000000, solver="adam", activation="tanh", hidden_layer_sizes=(5, 5), alpha=np.e**(min_score), random_state=1)
reg.fit(X_train, y_train)

# Predict
y_pred = reg.predict(X_test)

# MSE final
mean_squared_error(y_pred,y_test)


### Plot response curves

In [ ]:
# Create X_design_vec, which contains the median of each respective column
X_design = X.copy()
X_design_vec = pd.DataFrame(X_design.median()).transpose()
X_design_vec.head()

In [ ]:
# Set up a sequence for response variable to plot
min_res = min(X.loc[:,"weight"])
max_res = max(X.loc[:,"weight"])
seq = np.linspace(start=min_res,stop=max_res,num=50)

# Set up a list of moving resultants to plot
to_predict = []
for result in seq:
    X_design_vec.loc[0,"weight"] = result
    to_predict.append(X_design_vec.copy())

# Convert back to dataframe
to_predict = pd.concat(to_predict)

# Scale and predict
to_predict = scaler.transform(to_predict)
predictions = reg.predict(to_predict)

# Plot
plt.plot(seq,predictions)
plt.xlabel("weight (lbs)")
plt.ylabel("miles per gallon")
plt.title("mpg vs weight")
plt.show();

# Plot second curve

# Set up a sequence to plot
min_res = min(X.loc[:,"displacement"])
max_res = max(X.loc[:,"displacement"])
seq = np.linspace(start=min_res,stop=max_res,num=50)

# Set up a list of moving resultants to plot
to_predict = []
for result in seq:
    X_design_vec.loc[0,"displacement"] = result
    to_predict.append(X_design_vec.copy())

# Convert back to dataframe
to_predict = pd.concat(to_predict)

# Scale and predict
to_predict = scaler.transform(to_predict)
predictions = reg.predict(to_predict)

# Plot
plt.plot(seq,predictions)
plt.xlabel("displacement")
plt.ylabel("miles per gallon")
plt.title("mpg vs displacement")
plt.show();

Our model captures the inverse proportional relationship between both response variables weight and displacement
plotted against mpg. This is a positive outcome, as it means the model as applied to the validation dataset managed to
capture these signals in the data. We can therefore conclude that the model generalises well and that variance is
sufficiently low for this model to capture salient features in the underlying data.